# Подготовка датасета: Кот Бегемот

In [ ]:
!pip install -q openai

In [ ]:
import re, json, time
from openai import OpenAI

API_KEY = "sk-..."  # <-- вставь свой ключ OpenAI
client = OpenAI(api_key=API_KEY)
CHARACTER = "Кот Бегемот"

## 1. Загрузка и предобработка книги

In [ ]:
with open("/content/master_margarita.txt", "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

raw_text = raw_text.replace("\r", "\n")
raw_text = re.sub(r"\n{3,}", "\n\n", raw_text)

print(f"Загружено {len(raw_text)} символов")

## 2. Разбиение на чанки

In [ ]:
CHUNK_SIZE = 3000
OVERLAP = 200

chunks = []
start = 0
while start < len(raw_text):
    end = start + CHUNK_SIZE
    if end < len(raw_text):
        nl = raw_text.rfind("\n", start, end)
        if nl > start:
            end = nl
    chunks.append(raw_text[start:end])
    start = end - OVERLAP

print(f"Чанков: {len(chunks)}")

## 3. Извлечение реплик Бегемота через GPT-4o

In [ ]:
all_lines = []

for i, chunk in enumerate(chunks):
    print(f"Чанк {i+1}/{len(chunks)}...", end=" ")

    prompt = f"""Вот фрагмент романа «Мастер и Маргарита» Булгакова:

---
{chunk}
---

Извлеки ВСЕ реплики Кота Бегемота из этого фрагмента.
Бегемот может называться: Бегемот, кот, котище, котяра.
Включай только прямую речь Бегемота, без авторского текста.
Если реплик Бегемота нет — верни пустой массив.

Верни ТОЛЬКО JSON-массив строк:
["реплика 1", "реплика 2"]"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"^```json\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
        lines = json.loads(raw)
        all_lines.extend(lines)
        print(f"-> {len(lines)} реплик")
    except Exception as e:
        print(f"ОШИБКА: {e}")

    time.sleep(0.5)

unique_lines = list(dict.fromkeys(all_lines))
unique_lines = [l for l in unique_lines if len(l) > 10]

print(f"\nИзвлечено реплик Бегемота: {len(unique_lines)}")
for l in unique_lines[:10]:
    print(f"  - {l[:100]}")

## 4. Генерация пар вопрос-ответ через GPT-4o

In [ ]:
examples = "\n".join(f"- «{l}»" for l in unique_lines[:20])

TOPICS = [
    ["еда и напитки", "хулиганство и озорство", "философия жизни"],
    ["люди и их глупость", "магия и сверхъестественное", "культура и искусство"],
    ["дружба и верность", "деньги и жадность", "трусость и храбрость"],
    ["современные технологии", "работа и карьера", "путешествия"],
    ["любовь и отношения", "правда и ложь", "советы на каждый день"],
    ["погода", "мода и внешний вид", "домашние животные"],
]

augmented = []

for i, topics in enumerate(TOPICS):
    topics_str = ", ".join(topics)
    print(f"Батч {i+1}/{len(TOPICS)}: {topics_str}")

    prompt = f"""Ты — эксперт по персонажу {CHARACTER} из «Мастера и Маргариты» Булгакова.

Примеры его реплик:
{examples}

Сгенерируй 10 пар «вопрос человека → ответ в стиле {CHARACTER}».
Темы: {topics_str}.
Ответы: 1-3 предложения, на русском, в характерной хамоватой и остроумной манере Бегемота.

Верни ТОЛЬКО JSON-массив:
[{{"question": "...", "answer": "..."}}]"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.9,
        )
        raw = resp.choices[0].message.content.strip()
        raw = re.sub(r"^```json\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
        pairs = json.loads(raw)
        augmented.extend(pairs)
        print(f"  -> {len(pairs)} пар")
    except Exception as e:
        print(f"  ОШИБКА: {e}")

print(f"\nСгенерировано диалогов через LLM: {len(augmented)}")

## 5. Объединение и формат для SFTTrainer

In [ ]:
all_qa = []

generic_q = ["Что скажешь?", "И что же?", "Расскажи подробнее.",
             "Как так?", "Ты уверен?", "Что ты имеешь в виду?",
             "Продолжай.", "А дальше?", "Что думаешь?", "Ну и?"]

for i, line in enumerate(unique_lines):
    all_qa.append({
        "question": generic_q[i % len(generic_q)],
        "answer": line,
    })

all_qa.extend(augmented)

print(f"Из книги: {len(unique_lines)}, сгенерировано: {len(augmented)}, итого: {len(all_qa)}")

In [ ]:
SYSTEM = (
    f"Ты — {CHARACTER} из романа «Мастер и Маргарита» Булгакова. "
    f"Отвечай в его характерном стиле: дерзко, остроумно, с котовьей наглостью."
)
IM_END = "<|im_end|>"

sft_dataset = []
for p in all_qa:
    q, a = p["question"].strip(), p["answer"].strip()
    if not q or not a:
        continue
    sft_dataset.append({
        "prompt": f"<|im_start|>system\n{SYSTEM}{IM_END}\n<|im_start|>user\n{q}{IM_END}\n<|im_start|>assistant\n",
        "completion": f"{a}{IM_END}",
    })

print(f"Записей для SFT: {len(sft_dataset)}")

## 6. Сохранение

In [ ]:
with open("dataset_behemot_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_qa, f, ensure_ascii=False, indent=2)

with open("dataset_behemot_sft.json", "w", encoding="utf-8") as f:
    json.dump(sft_dataset, f, ensure_ascii=False, indent=2)

print("Сохранено: dataset_behemot_raw.json, dataset_behemot_sft.json")